# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Maaz89/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Unit of Analysis / Grain: One row = One unique web page (url_hash) for a specific client (client_id) evaluated at a specific month snapshot.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Table(s) Used: The primary warehouse dataset on Hugging Face (FlyRank/internship-warehouse), specifically slicing mid-panel month month=2026-03.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*



In [7]:
import duckdb
import os
from google.colab import userdata

# 1. Fetch token from Colab Secrets
try:
    hf_token = userdata.get('HF_TOKEN')
    os.environ["HF_TOKEN"] = hf_token
except Exception as e:
    print("Warning: HF_TOKEN secret not found in Colab Secrets panel.")
    hf_token = None

# 2. Initialize DuckDB
con = duckdb.connect()

# 3. Set configuration parameters (S3 endpoint) *before* loading httpfs
# Removed 'custom_user_agent' as it causes an error when set after connection.
con.execute("SET s3_endpoint='hf.co';")

# 4. Load httpfs extension
con.execute("INSTALL httpfs; LOAD httpfs;")

# The HF_TOKEN environment variable is automatically used by httpfs for authentication with hf:// URLs.
DATA_URL = "hf://datasets/FlyRank/internship-warehouse/data/month=2026-03/*.parquet"

print("======================================================================")
print("1. VERIFY GRAIN (Claim: One row = One URL per Client in Month Snapshot)")
print("======================================================================")
q_grain = f"""
SELECT
    COUNT(*) as total_rows,
    COUNT(DISTINCT url_hash) as unique_urls,
    COUNT(DISTINCT client_id) as unique_clients,
    COUNT(DISTINCT (client_id || '_' || url_hash)) as unique_grain_keys,
    CASE
        WHEN COUNT(*) = COUNT(DISTINCT (client_id || '_' || url_hash))
        THEN 'PASSED: Grain is strictly unique per (client_id, url_hash)'
        ELSE 'FAILED: Duplicate rows found for the primary key'
    END as grain_check_result
FROM '{DATA_URL}'
"""
df_grain = con.execute(q_grain).df()
print(df_grain.to_string(index=False))


print("\n======================================================================")
print("2. VERIFY COUNTS & SLICE SIZE (Claim: Mid-Panel Month 2026-03 Dataset)")
print("======================================================================")
q_counts = f"""
SELECT
    COUNT(*) as total_row_count,
    COUNT(DISTINCT client_id) as total_clients,
    SUM(CASE WHEN is_declining_label = 1 THEN 1 ELSE 0 END) as declining_target_count,
    ROUND(AVG(CASE WHEN is_declining_label = 1 THEN 1.0 ELSE 0.0 END) * 100, 2) as target_imbalance_pct
FROM '{DATA_URL}'
"""
df_counts = con.execute(q_counts).df()
print(df_counts.to_string(index=False))


print("\n======================================================================")
print("3. VERIFY AVAILABILITY & MISSING VALUES (Claim: IS TRUE Validity Filter)")
print("======================================================================")
q_availability = f"""
SELECT
    COUNT(*) as total_raw_rows,
    COUNT(CASE WHEN is_valid IS TRUE THEN 1 END) as valid_surviving_rows,
    COUNT(*) - COUNT(CASE WHEN is_valid IS TRUE THEN 1 END) as invalid_discarded_rows,
    SUM(CASE WHEN impressions_90d IS NULL THEN 1 ELSE 0 END) as null_impressions,
    SUM(CASE WHEN engagement_rate IS NULL THEN 1 ELSE 0 END) as null_engagement,
    SUM(CASE WHEN avg_position IS NULL THEN 1 ELSE 0 END) as null_position
FROM '{DATA_URL}'
"""
df_availability = con.execute(q_availability).df()
print(df_availability.to_string(index=False))


print("\n======================================================================")
print("4. VERIFY TEMPORAL WINDOW (Claim: Historical Snapshot Boundaries)")
print("======================================================================")
q_window = f"""
SELECT
    MIN(snapshot_date) as earliest_snapshot_date,
    MAX(snapshot_date) as latest_snapshot_date,
    COUNT(DISTINCT snapshot_date) as distinct_snapshot_dates
FROM '{DATA_URL}'
"""
df_window = con.execute(q_window).df()
print(df_window.to_string(index=False))

1. VERIFY GRAIN (Claim: One row = One URL per Client in Month Snapshot)


HTTPException: HTTP Error: HTTP GET error on 'https://huggingface.co/api/datasets/FlyRank/internship-warehouse/tree/main/data/month=2026-03' (HTTP 404)

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

While the warehouse dataset provides a clean historical baseline, operating in production requires acknowledging the structural blind spots of this specific slice. This data cannot capture or evaluate three primary limitations:

1. Unbalanced Client History & Non-Stationary Drift
The Reality: Domains in the panel range from enterprise sites with 5+ years of continuous tracking to newer clients onboarded mid-year.

The Blind Spot: Aggregated features like impressions_90d treat all domains as if they exist on a mature, stationary baseline. The data cannot distinguish between true content decay and artificial noise caused by domain authority shifts, major site migrations, or sudden tracking script additions midway through the historical window.

2. Partial Attribution (GSC-Only Early Rows)
The Reality: Google Search Console (GSC) metrics (impressions, avg position) and Google Analytics 4 (GA4) metrics (engagement rate, session duration) are synced across different timelines and APIs.

The Blind Spot: In earlier snapshot rows, GA4 integrations may be missing or delayed while GSC query data is present. The model cannot observe post-click user intent or page satisfaction for these rows; it sees only search engine visibility signals, creating a blind spot around bounce-driven traffic drops versus true algorithm-driven rank drops.

3. Window Overlaps & Temporal Autocorrelation
The Reality: Features rely on rolling 90-day aggregations, while snapshot slices are drawn in monthly intervals (month=2026-03, month=2026-04, etc.).

The Blind Spot: Successive monthly snapshots share approximately 60 days of identical underlying Search Console records. The model cannot treat sequential rows as independent, identically distributed (i.i.d.) observations. Standard random K-fold cross-validation will drastically underestimate prediction error due to temporal leakage across overlapping feature windows.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.